In [55]:
import pandas as pd
import glob as glob
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
import gpplot as gpp
import random
pd.options.mode.chained_assignment = None

Modeled off of code by Smriti Srikanth

Running the MAGeCK-MLE and MAGeCK-RRA hit calling methods on Jacquere A375 data (with use of pRDA_734 vector)

Using MAGeCK v. 0.5.9.4

In [56]:
#importing data from data cleaned for comparison of Jacquere vs. Brunello screen performance 
screen_results=pd.read_csv("../../6. Jacquere Screen Performance/Data/jacquereA375lfc.csv")
screen_results

,Unnamed: 0,Construct Barcode,condition,sgRNA_lfc,Target Gene ID,Source,Target Gene Symbol,Target Transcript,Reference Sequence,Strand of sgRNA,sgRNA Cut Position (1-based),Exon Number,Aggregate CFD Score,Off-Target CFD100 Hits,Off-Target Tier I CFD100 Hits,On-Target Efficacy Score
0,0,AAAAAAGAAAGGATACGTCC,Jacquere A375,-0.770787,ENSG00000189283,GENCODE,FHIT,ENST00000492590.6,NC_000003.12,+,60536861.0,5.0,0.5714,0.0,0.0,1.1390
1,2,AAAAACAGCTGATGAAACCT,Jacquere A375,-0.323128,ENSG00000116815,GENCODE,CD58,ENST00000369489.10,NC_000001.11,+,116544604.0,2.0,0.8133,0.0,0.0,0.5047
2,4,AAAAACCTGGGAAAGATATA,Jacquere A375,0.270047,ENSG00000188295,GENCODE,ZNF669,ENST00000448299.7,NC_000001.11,-,247101736.0,3.0,1.5184,0.0,0.0,0.9580
3,6,AAAAAGAGGCGGCATTACAG,Jacquere A375,-0.251598,ENSG00000273957,GENCODE,NaN,ENST00000614465.2,NC_000005.10,+,100389890.0,1.0,0.6050,0.0,0.0,0.6677
4,8,AAAAAGATAGATGTGCTAAA,Jacquere A375,-3.443730,ENSG00000151304,GENCODE,SRFBP1,ENST00000339397.5,NC_000005.10,+,122019337.0,5.0,0.0000,0.0,0.0,0.3718
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
65791,131582,TTTGTGGAGCCGGTACATGA,Jacquere A375,0.459668,ENSG00000033100,GENCODE,CHPF2,ENST00000035307.7,NC_000007.14,-,151236520.0,3.0,0.2857,0.0,0.0,1.2660
65792,131584,TTTGTGGGTACATCTAACCC,Jacquere A375,0.202400,ONE_SITE_INTERGENIC,ONE_SITE_INTERGENIC,ONE_SITE_INTERGENIC,NaN,NaN,NaN,NaN,NaN,0.3333,NaN,NaN,1.0350
65793,131586,TTTGTGGTAGGAGTAGGTGT,Jacquere A375,-1.098735,ENSG00000161999,GENCODE,JMJD8,ENST00000609261.6,NC_000016.10,+,683703.0,4.0,1.6667,0.0,0.0,0.9859
65794,131588,TTTGTGTATTACCTCAGGAA,Jacquere A375,-0.843163,ENSG00000189252,GENCODE,SPANXN3,ENST00000370503.2,NC_000023.11,-,143509068.0,2.0,0.7143,0.0,0.0,0.4334


In [57]:
num_pseudogene = 1000
genetargeting_results = (screen_results.loc[(screen_results['Target Gene Symbol']
                                            .str.contains('ONE_SITE_INTERGENIC')==False) &
                                              (screen_results['Target Gene Symbol']
                                            .str.contains('NO_SITE')==False) &
                                              (screen_results['Target Gene Symbol']
                                            .str.contains('MULTIPLE_INTERGENIC')==False),:]
                      .reset_index(drop=True))
# number of guides per pseudogene should match that of actual genes
n_guides_per_gene = genetargeting_results.groupby('Target Gene Symbol')['Construct Barcode'].nunique().mode()[0]
control_sgrnas = screen_results[screen_results["Target Gene Symbol"]=="ONE_SITE_INTERGENIC"]['Construct Barcode'].unique()
pseudogene_df_list = []

random.seed = 0
for i in range(num_pseudogene):
     chosen_guides = random.choices(control_sgrnas, k=n_guides_per_gene)
     temp_df_list = []
     for j in range(n_guides_per_gene):
         g = chosen_guides[j]
         temp_row = screen_results.loc[(screen_results['Construct Barcode'] == g) &
                                           (screen_results['Target Gene Symbol'].str.contains('ONE_SITE_INTERGENIC')),:]
         temp_row.loc[:,'Construct Barcode with identifier'] = g + 'p' + str(i) + 'g' + str(j)
         temp_df_list.append(temp_row)
     temp_df = pd.concat(temp_df_list)
     temp_df.loc[:,'Target Gene Symbol'] = 'Pseudogene_' + str(i)
     pseudogene_df_list.append(temp_df)

pseudogene_df = pd.concat(pseudogene_df_list).reset_index(drop=True)
screen_results_withpseudogenes = pd.concat([screen_results,pseudogene_df]).reset_index(drop=True)

screen_results_withpseudogenes["Construct Barcode with identifier"]=screen_results_withpseudogenes.apply(lambda x: x["Construct Barcode"] if type(x["Construct Barcode with identifier"])==float else x["Construct Barcode with identifier"],axis=1)
#remove NA target gene symbols
screen_results_withpseudogenes["Target Gene Symbol"]=screen_results_withpseudogenes.apply(lambda x:x["Target Gene ID"] if type(x["Target Gene Symbol"])==float else x["Target Gene Symbol"],axis=1)

### Generating input files
Must feature raw read counts for each construct along with its corresponding target.

In [59]:
mageck_guide_gene_annotation = screen_results_withpseudogenes[['Construct Barcode','Construct Barcode with identifier','Target Gene Symbol']].drop_duplicates()

#read in raw results
reads_df =pd.read_table("../../6. Jacquere Screen Performance/Data/counts-EK20250627_Jacquere_allinone.txt")

#must merge on the guide sequence without pseudogene identifer appended because that is what is present in reads_df
annotated_reads_df = mageck_guide_gene_annotation.merge(reads_df,how = 'left',on = 'Construct Barcode')

#dropping A549 columns since we are only interested in analyzing A375 data for now
annotated_reads_df = annotated_reads_df.drop(['Construct IDs','A549_CP0082_repA','A549_CP0082_repB','Empty'],axis=1)

#MAGeCK throws an error when the same guide targets multiple genes, so make construct barcode unique by appending gene symbol
annotated_reads_df['Construct_Barcode_with_identifier'] = annotated_reads_df['Construct Barcode with identifier'] + annotated_reads_df['Target Gene Symbol']
#also throws an error if column names have spaces
annotated_reads_df=annotated_reads_df.rename(columns={"Target Gene Symbol":"target_gene"})

annotated_reads_df[['Construct_Barcode_with_identifier','target_gene','A375_CP0082_repA','A375_CP0082_repB',"CP0082_pDNA"]].to_csv('../Data/hitcalling_methods_inputs/mageck_input_jacquere_A375counts.txt', sep='\t', index=False)

control_sgrnas_raw = annotated_reads_df[annotated_reads_df["Construct Barcode"].isin(control_sgrnas)]
control_sgrnas_raw=control_sgrnas_raw[["Construct Barcode"]].drop_duplicates()
control_sgrnas_raw["Construct Barcode"]=control_sgrnas_raw["Construct Barcode"]+"ONE_SITE_INTERGENIC"
control_sgrnas_raw[['Construct Barcode']].drop_duplicates().to_csv('../Data/hitcalling_methods_inputs/mageck_input_jacquere_controls_A375.txt', index=False, header = False)



### Running MAGeCK-RRA

Normalizing guides to the set of all guides (--norm-method total) for consistency with the approach implemented for the Z-score method pipeline

mageck test -k mageck_input_jacquere_A375counts.txt -t A375_CP0082_repA,A375_CP0082_repB -c CP0082_pDNA -n jacquere --control-sgrna mageck_input_jacquere_controls_A375.txt --norm-method total -n jacquere_A375_mageck_RRA_results 

### Running MAGeCK-MLE 
Skipping the permutation step, which determines empirical p-values with the set of all targeting sgRNAs as the null distribution, since I am computing this manually just using the intergenic controls as the null distribution. 

mageck mle -k mageck_input_jacquere_A375counts.txt --day0-label CP0082_pDNA --control-sgrna mageck_input_jacquere_controls_A375.txt --norm-method total -n jacquere_A375_mageck_MLE_results --no-permutation-by-group